In [2]:
import pandas as pd
import os
import sys
import re
import pickle
from sklearn.dummy import DummyClassifier
sys.path.append("/home/onyxia/work/WP10_Cluster1_StatCodGen/statcodgen")
from structured import Structured
from codifier import Codifier

# Dummy Codifier Demo

## Mock-up Structure

In [4]:
class StructuredMockup(Structured): 
    """
    Structured hierarchy for demo purposes

    Levels are determined strictly by the length of the classification code.
        
    """
    def get_level(self, code): # Structured child class specific to WZ
        """
        Structured hierarchy for German WZ
    
        Parameters
        ---------
        code : str
            WZ classification code.
    
        Returns
        -------
        int
            Hierarchical level based on code length
        """
        code = str(code).replace('.', '')      
        return len(code)-1

In [5]:
# The dummy classification with one level and three classes
df = pd.DataFrame(data=
                  {"code": ['1', '2', '3', '4', '5'], 
                   "text": ['Wohnen', 'Aktivitaet', 'Nahrungsmittel', 'Fahrzeug', 'Sonstiges']}
                 )

In [6]:
# Generate the instance of StructuredMockup
mock_structure = StructuredMockup(
    structure_df=df,
    names_l=['Class']
)
mock_structure.level_dict

{0: 'Class'}

In [7]:
# Load to check the structure
mus = mock_structure.load_structue(df)
mus

,code,text,level
0,1,Wohnen,0
1,2,Aktivitaet,0
2,3,Nahrungsmittel,0
3,4,Fahrzeug,0
4,5,Sonstiges,0


## Dummy Codifier

In [8]:
# Now we create a Codifier's son with a method that always predicts the majority class and assigns 0/1 confidence scores.
class CodifierDummy(Codifier):
    def save(self, name):
        with open(os.path.join(self.root_path, name),'wb') as f:
            pickle.dump(self.model,f)
        
    def load(self, name):
        with open(os.path.join(self.root_path, name),'rb') as f:
            self.model = pickle.load(f)
        
    def train(self, **kwargs):
        train_set = kwargs.get('train_set', self.train_df)
        train_data = self.get_train_dataset(train_set, name='test')
        
        dummy_clf = DummyClassifier(strategy="most_frequent")
        self.model = dummy_clf.fit(train_data['text'], train_data['label'])
        
    def get_pred_for_batch(self, samples, idxs, clean_samples=False):
        """
        Generate hierarchical predictions for a batch of samples with random probabilities,
        ensuring that upper levels and their children are consistently sorted.
        
        Parameters
        ----------
        samples : list of str
            Text samples to predict.
        idxs : list of int
            Indices corresponding to each sample.
        clean_samples : bool, optional (default=False)
            Whether to preprocess the samples before prediction.
        
        Returns
        -------
        dict
            Dictionary mapping each index to a hierarchical prediction with sorted labels and confidences.
        """
        # if clean_samples:
        #     samples = [preprocess_text(s) for s in samples]

        last_level_labels = list(self.structure.reversed_hierarchy.keys())

        results = dict()

        preds_raw = self.model.predict(samples)
        probs = self.model.predict_proba(samples)

        for idx in idxs:
            results[idx] = {}
            results[idx][f'label_{self.structure.level_dict[0]}'] = last_level_labels
            results[idx][f'conf_{self.structure.level_dict[0]}'] = probs

        return results        

In [9]:
# Train and test data for the mockup structure
train_df = pd.DataFrame({'label': ['1', '2', '3', '4', '2', '5'], 
                         'text': ['Huette', 'schwimmen', 'Pizza', 'Bus', 'schreiben', 'blabla']}
                       ) 
test_df = pd.DataFrame({'label': ['1', '2', '3', '4'], 
                        'text': ['Villa', 'rennen', 'Nudeln', 'Auto'], 'source': ['s1', 's1', 's1', 's1']}
                      ) 

In [10]:
cd = CodifierDummy(
    structure_instance=mock_structure, 
    train_df=train_df, 
    test_df=test_df, 
    root_path="./test", 
    min_lenght_texts=1
)

2026-02-09 08:50:45,412 - INFO - Loading train_df..
2026-02-09 08:50:45,414 - INFO - train_df raw data count: 6
2026-02-09 08:50:45,418 - INFO - train_df bad codes count: 0
2026-02-09 08:50:45,419 - INFO - train_df bad descriptions count: 0
2026-02-09 08:50:45,421 - INFO - train_df pruned data count: 6
2026-02-09 08:50:45,422 - INFO - Loading test_df..
2026-02-09 08:50:45,423 - INFO - test_df raw data count: 4
2026-02-09 08:50:45,426 - INFO - test_df bad codes count: 0
2026-02-09 08:50:45,427 - INFO - test_df bad descriptions count: 0
2026-02-09 08:50:45,429 - INFO - test_df pruned data count: 4


In [11]:
# Train and save a model or load a pretrained one

cd.train()
cd.save(name='test')
#cd.load(name='test')

2026-02-09 08:50:46,826 - INFO - Loading test..
2026-02-09 08:50:46,828 - INFO - test raw data count: 6
2026-02-09 08:50:46,831 - INFO - test bad codes count: 0
2026-02-09 08:50:46,832 - INFO - test bad descriptions count: 0
2026-02-09 08:50:46,833 - INFO - test pruned data count: 6


In [12]:
# Predict some samples in batch mode

samples = ["tanzen", "Flugzeug"]
idxs = [0, 1]

cd.get_pred_for_batch(samples=samples, idxs=idxs)

{0: {'label_Class': ['1', '2', '3', '4', '5'],
  'conf_Class': array([[0., 1., 0., 0., 0.],
         [0., 1., 0., 0., 0.]])},
 1: {'label_Class': ['1', '2', '3', '4', '5'],
  'conf_Class': array([[0., 1., 0., 0., 0.],
         [0., 1., 0., 0., 0.]])}}

In [13]:
cd.evaluate(version='example', get_curve=False) # Some errors when producing the curve

2026-02-09 08:50:52,685 - INFO - Test set size 4 for source all
2026-02-09 08:50:52,687 - INFO - test_set_all has 1 codes unrepresented
Predicting test_set for 1 classes: 100%|██████████| 4/4 [00:00<00:00, 2533.94it/s]


{'Class': {'accuracy': 0.25,
  'f1_score': {'micro': 0.25, 'macro': 0.1, 'weighted': 0.1}}}

In [17]:
cd.predict(desc_l=['test', 'blah', 'blubb'],
           mode='assistance',#'codification',
           hierarchical_level=1,
           threshold=0.01
          )

ValueError: Per-column arrays must each be 1-dimensional